# Inspect Political-Corruption Topic Results

This notebook reads the final topic-model outputs and GPT topic labels, then produces human-readable tables and interactive country/time visualizations.

Expected inputs from the topic workflow:

- `topic_info.csv`
- `document_topics.csv.gz`
- `topic_labels_llm.csv`

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_PATH = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in [NOTEBOOK_PATH, *NOTEBOOK_PATH.parents] if (path / "config.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)

DEFAULT_TOPIC_DIR = Path(
    "/home/akroon/data/1t_storage/RESPOND-victims-of-corruption/"
    "topic_classification/bertopic_political_corruption_granular"
)

TOPIC_DIR = DEFAULT_TOPIC_DIR
TOP_N_TOPICS = 12
TIME_UNIT = "year"  # "year" or "month"
ANALYSIS_LABEL_COLUMN = "topic_short_label"  # Inductive GPT label for raw BERTopic clusters.
RAW_TOPIC_LABEL_COLUMN = "topic_short_label"
LIFT_FLOOR = 0.005  # Hide labels with very small baseline shares in lift plots.

TOPIC_INFO_PATH = TOPIC_DIR / "topic_info.csv"
DOCUMENT_TOPICS_PATH = TOPIC_DIR / "document_topics.csv.gz"
TOPIC_LABELS_PATH = TOPIC_DIR / "topic_labels_llm.csv"

TOPIC_DIR

## Load Results

In [ ]:
missing = [path for path in [TOPIC_INFO_PATH, DOCUMENT_TOPICS_PATH] if not path.exists()]
if missing:
    raise FileNotFoundError("Missing topic output(s): " + ", ".join(str(path) for path in missing))

topic_info = pd.read_csv(TOPIC_INFO_PATH)
document_topics = pd.read_csv(DOCUMENT_TOPICS_PATH)

if TOPIC_LABELS_PATH.exists():
    topic_labels = pd.read_csv(TOPIC_LABELS_PATH)
else:
    topic_labels = pd.DataFrame()
    print(f"No GPT topic labels found yet: {TOPIC_LABELS_PATH}")

print(f"Topic directory: {TOPIC_DIR}")
print(f"Topics: {len(topic_info):,}")
print(f"Document-topic rows: {len(document_topics):,}")
print(f"GPT-labelled topics: {len(topic_labels):,}")

In [ ]:
def build_topic_lookup(topic_info, topic_labels):
    labels = topic_info[["Topic", "Name", "Count"]].copy()
    if not topic_labels.empty:
        keep_cols = [
            col for col in [
                "Topic",
                "llm_topic_label",
                "llm_topic_short_label",
                "llm_primary_domain",
                "llm_secondary_domain",
                "llm_generic_domain_label",
                "llm_generic_domain_short_label",
                "llm_corruption_type",
                "llm_country_event_specific",
                "llm_domain_evidence",
                "llm_cross_country_comparability",
                "llm_label_rationale",
                "llm_topic_summary",
                "llm_inclusion_rule",
                "llm_exclusion_rule",
                "llm_confidence",
            ]
            if col in topic_labels.columns
        ]
        labels = labels.merge(topic_labels[keep_cols], on="Topic", how="left")

    labels["topic_label"] = labels.get("llm_topic_label", pd.Series(index=labels.index, dtype=object))
    labels["topic_label"] = labels["topic_label"].fillna("").astype(str)
    labels.loc[labels["topic_label"].str.strip().eq(""), "topic_label"] = labels["Name"]

    labels["topic_short_label"] = labels.get("llm_topic_short_label", pd.Series(index=labels.index, dtype=object))
    labels["topic_short_label"] = labels["topic_short_label"].fillna("").astype(str)
    labels.loc[labels["topic_short_label"].str.strip().eq(""), "topic_short_label"] = labels["topic_label"]

    labels["primary_domain"] = labels.get("llm_primary_domain", pd.Series(index=labels.index, dtype=object))
    labels["primary_domain"] = labels["primary_domain"].fillna("").astype(str)
    labels.loc[labels["primary_domain"].str.strip().eq(""), "primary_domain"] = labels.get("llm_corruption_type", labels["topic_label"])
    labels["primary_domain"] = labels["primary_domain"].fillna(labels["topic_label"])

    labels["secondary_domain"] = labels.get("llm_secondary_domain", pd.Series(index=labels.index, dtype=object))
    labels["secondary_domain"] = labels["secondary_domain"].fillna("none").astype(str)

    labels["generic_domain_label"] = labels.get("llm_generic_domain_label", pd.Series(index=labels.index, dtype=object))
    labels["generic_domain_label"] = labels["generic_domain_label"].fillna("").astype(str)
    labels.loc[labels["generic_domain_label"].str.strip().eq(""), "generic_domain_label"] = labels["primary_domain"]
    labels["generic_domain_label"] = labels["generic_domain_label"].fillna(labels["topic_label"])

    labels["generic_domain_short_label"] = labels.get("llm_generic_domain_short_label", pd.Series(index=labels.index, dtype=object))
    labels["generic_domain_short_label"] = labels["generic_domain_short_label"].fillna("").astype(str)
    labels.loc[labels["generic_domain_short_label"].str.strip().eq(""), "generic_domain_short_label"] = labels["generic_domain_label"]
    return labels

topic_lookup = build_topic_lookup(topic_info, topic_labels)
docs = document_topics.merge(topic_lookup, left_on="topic", right_on="Topic", how="left")
docs["topic_label"] = docs["topic_label"].fillna(docs["topic"].astype(str))
docs["topic_short_label"] = docs["topic_short_label"].fillna(docs["topic_label"])
for optional_col, fallback_col in [
    ("primary_domain", "topic_label"),
    ("secondary_domain", None),
    ("generic_domain_label", "topic_label"),
    ("generic_domain_short_label", "topic_short_label"),
]:
    if optional_col not in docs.columns:
        docs[optional_col] = "none" if fallback_col is None else docs[fallback_col]
    elif fallback_col is None:
        docs[optional_col] = docs[optional_col].fillna("none")
    else:
        docs[optional_col] = docs[optional_col].fillna(docs[fallback_col])
if ANALYSIS_LABEL_COLUMN not in docs.columns:
    raise ValueError(f"ANALYSIS_LABEL_COLUMN not found: {ANALYSIS_LABEL_COLUMN}")
if "analysis_weight" not in docs.columns:
    docs["analysis_weight"] = 1.0

non_outlier_docs = docs[docs["topic"].ne(-1)].copy()
topic_lookup.head()

## Topic Overview

In [ ]:
topic_totals = (
    non_outlier_docs.groupby([ANALYSIS_LABEL_COLUMN], dropna=False)["analysis_weight"]
    .sum()
    .rename("weighted_articles")
    .reset_index()
    .sort_values("weighted_articles", ascending=False)
)
topic_totals["weighted_share"] = topic_totals["weighted_articles"] / topic_totals["weighted_articles"].sum()
top_labels = topic_totals.head(TOP_N_TOPICS)[ANALYSIS_LABEL_COLUMN].tolist()

display(topic_totals.head(30))

fig = px.bar(
    topic_totals.head(30).sort_values("weighted_articles"),
    x="weighted_articles",
    y=ANALYSIS_LABEL_COLUMN,
    orientation="h",
    title="Largest Political-Corruption Topics",
    labels={"weighted_articles": "Weighted articles", ANALYSIS_LABEL_COLUMN: "Topic"},
)
fig.update_layout(height=800)
fig.show()

In [ ]:
summary_cols = [
    col for col in [
        "Topic",
        "Count",
        "topic_label",
        "primary_domain",
        "secondary_domain",
        "topic_short_label",
        "llm_cross_country_comparability",
        "llm_country_event_specific",
        "llm_label_rationale",
        "llm_topic_summary",
        "generic_domain_label",
        "generic_domain_short_label",
        "llm_corruption_type",
        "llm_domain_evidence",
        "llm_inclusion_rule",
        "llm_exclusion_rule",
        "llm_confidence",
        "Name",
    ]
    if col in topic_lookup.columns
]

display(
    topic_lookup[topic_lookup["Topic"].ne(-1)]
    .sort_values("Count", ascending=False)
    [summary_cols]
    .head(30)
)

## Cross-Country Topic Composition

In [ ]:
docs_top = non_outlier_docs[non_outlier_docs[ANALYSIS_LABEL_COLUMN].isin(top_labels)].copy()

country_topic = (
    docs_top.groupby(["country", ANALYSIS_LABEL_COLUMN], dropna=False)["analysis_weight"]
    .sum()
    .rename("weighted_articles")
    .reset_index()
)
country_topic["share"] = country_topic["weighted_articles"] / country_topic.groupby("country")["weighted_articles"].transform("sum")

heatmap_data = country_topic.pivot(index="country", columns=ANALYSIS_LABEL_COLUMN, values="share").fillna(0)
fig = px.imshow(
    heatmap_data,
    aspect="auto",
    color_continuous_scale="Viridis",
    labels={"color": "Within-country share"},
    title=f"Top {TOP_N_TOPICS} Political-Corruption Topics By Country",
)
fig.update_layout(height=650)
fig.show()

## Country Topic Lift Diagnostics

Shares can look flat when topic prevalence is similar. Lift shows where a country over- or under-represents a topic compared with the overall sample baseline. Values are log2 lift: `1` means twice the baseline share, `-1` means half the baseline share.

In [ ]:
overall_label_share = (
    docs_top.groupby(ANALYSIS_LABEL_COLUMN, dropna=False)["analysis_weight"]
    .sum()
    .div(docs_top["analysis_weight"].sum())
    .rename("overall_share")
    .reset_index()
)

country_topic_lift = country_topic.merge(overall_label_share, on=ANALYSIS_LABEL_COLUMN, how="left")
country_topic_lift = country_topic_lift[country_topic_lift["overall_share"].ge(LIFT_FLOOR)].copy()
country_topic_lift["lift"] = country_topic_lift["share"] / country_topic_lift["overall_share"]
country_topic_lift["log2_lift"] = np.log2(country_topic_lift["lift"].replace(0, np.nan))

lift_heatmap = country_topic_lift.pivot(index="country", columns=ANALYSIS_LABEL_COLUMN, values="log2_lift").fillna(0)
fig = px.imshow(
    lift_heatmap,
    aspect="auto",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    labels={"color": "log2 lift"},
    title=f"Country Over/Under-Representation Of Top {TOP_N_TOPICS} Topics",
)
fig.update_layout(height=650)
fig.show()

display(
    country_topic_lift.sort_values("log2_lift", ascending=False)
    [["country", ANALYSIS_LABEL_COLUMN, "share", "overall_share", "lift", "log2_lift", "weighted_articles"]]
    .head(30)
)


## Topic Shares Over Time

In [ ]:
trend_docs = docs_top.copy()

if TIME_UNIT == "month":
    if "date_parsed" not in trend_docs.columns:
        raise ValueError("Monthly trends require date_parsed in document_topics.csv.gz.")
    trend_docs["date_parsed"] = pd.to_datetime(trend_docs["date_parsed"], errors="coerce", utc=True)
    trend_docs["period"] = trend_docs["date_parsed"].dt.tz_convert(None).dt.to_period("M").astype(str)
else:
    trend_docs["period"] = trend_docs["year"].astype("Int64").astype(str)

time_topic = (
    trend_docs.groupby(["period", ANALYSIS_LABEL_COLUMN], dropna=False)["analysis_weight"]
    .sum()
    .rename("weighted_articles")
    .reset_index()
)
time_topic["share"] = time_topic["weighted_articles"] / time_topic.groupby("period")["weighted_articles"].transform("sum")

fig = px.area(
    time_topic.sort_values("period"),
    x="period",
    y="share",
    color=ANALYSIS_LABEL_COLUMN,
    title=f"Top {TOP_N_TOPICS} Political-Corruption Topic Shares Over Time",
    labels={"period": TIME_UNIT.title(), "share": "Topic share", ANALYSIS_LABEL_COLUMN: "Topic"},
)
fig.update_layout(height=650, hovermode="x unified")
fig.show()

## Time Topic Lift Diagnostics

This shows which topics become unusually prominent in particular years or months, relative to their overall baseline.

In [ ]:
time_topic_lift = time_topic.merge(overall_label_share, on=ANALYSIS_LABEL_COLUMN, how="left")
time_topic_lift = time_topic_lift[time_topic_lift["overall_share"].ge(LIFT_FLOOR)].copy()
time_topic_lift["lift"] = time_topic_lift["share"] / time_topic_lift["overall_share"]
time_topic_lift["log2_lift"] = np.log2(time_topic_lift["lift"].replace(0, np.nan))

time_lift_heatmap = time_topic_lift.pivot(index="period", columns=ANALYSIS_LABEL_COLUMN, values="log2_lift").fillna(0)
fig = px.imshow(
    time_lift_heatmap,
    aspect="auto",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    labels={"color": "log2 lift"},
    title=f"Over/Under-Representation Of Top {TOP_N_TOPICS} Topics Over Time",
)
fig.update_layout(height=650)
fig.show()

display(
    time_topic_lift.sort_values("log2_lift", ascending=False)
    [["period", ANALYSIS_LABEL_COLUMN, "share", "overall_share", "lift", "log2_lift", "weighted_articles"]]
    .head(30)
)


## Country Trends Over Time

In [ ]:
country_time = (
    trend_docs.groupby(["country", "period", ANALYSIS_LABEL_COLUMN], dropna=False)["analysis_weight"]
    .sum()
    .rename("weighted_articles")
    .reset_index()
)

fig = px.line(
    country_time.sort_values("period"),
    x="period",
    y="weighted_articles",
    color=ANALYSIS_LABEL_COLUMN,
    facet_row="country",
    title=f"Top {TOP_N_TOPICS} Political-Corruption Topic Volume By Country Over Time",
    labels={"period": TIME_UNIT.title(), "weighted_articles": "Weighted articles", ANALYSIS_LABEL_COLUMN: "Topic"},
    height=1400,
)
fig.update_yaxes(matches=None)
fig.show()

## Raw Topic Specificity Diagnostics

These tables flag clusters dominated by one country and show how raw BERTopic clusters map onto the plotted inductive topic labels.

In [ ]:
raw_country = (
    non_outlier_docs.groupby([RAW_TOPIC_LABEL_COLUMN, "country"], dropna=False)["analysis_weight"]
    .sum()
    .rename("weighted_articles")
    .reset_index()
)
raw_country["topic_total"] = raw_country.groupby(RAW_TOPIC_LABEL_COLUMN)["weighted_articles"].transform("sum")
raw_country["country_share_within_raw_topic"] = raw_country["weighted_articles"] / raw_country["topic_total"]
raw_specificity = (
    raw_country.groupby(RAW_TOPIC_LABEL_COLUMN, dropna=False)
    .agg(
        weighted_articles=("topic_total", "first"),
        max_country_share=("country_share_within_raw_topic", "max"),
        countries_ge_5pct=("country_share_within_raw_topic", lambda s: int((s >= 0.05).sum())),
    )
    .reset_index()
    .sort_values("max_country_share", ascending=False)
)

display(raw_specificity.head(30))

raw_to_label = (
    non_outlier_docs.groupby([RAW_TOPIC_LABEL_COLUMN, ANALYSIS_LABEL_COLUMN], dropna=False)["analysis_weight"]
    .sum()
    .rename("weighted_articles")
    .reset_index()
    .sort_values("weighted_articles", ascending=False)
)
display(raw_to_label.head(50))


## Country-Time Lift

This is the strongest variance check: within each country-year, which topics are unusually prominent relative to their overall baseline?

In [ ]:
country_time_share = country_time.copy()
country_time_share["share"] = country_time_share["weighted_articles"] / country_time_share.groupby(["country", "period"])["weighted_articles"].transform("sum")
country_time_lift = country_time_share.merge(overall_label_share, on=ANALYSIS_LABEL_COLUMN, how="left")
country_time_lift = country_time_lift[country_time_lift["overall_share"].ge(LIFT_FLOOR)].copy()
country_time_lift["lift"] = country_time_lift["share"] / country_time_lift["overall_share"]
country_time_lift["log2_lift"] = np.log2(country_time_lift["lift"].replace(0, np.nan))

fig = px.line(
    country_time_lift.sort_values("period"),
    x="period",
    y="log2_lift",
    color=ANALYSIS_LABEL_COLUMN,
    facet_row="country",
    title=f"Country-Time Topic Lift For Top {TOP_N_TOPICS} Topics",
    labels={"period": TIME_UNIT.title(), "log2_lift": "log2 lift", ANALYSIS_LABEL_COLUMN: "Topic"},
    height=1400,
)
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.update_yaxes(matches=None)
fig.show()


## Inspect Example Articles By Topic

In [ ]:
SELECT_LABEL = top_labels[0] if top_labels else None
N_EXAMPLES = 10

example_cols = [col for col in ["country", "year", "date_parsed", "prob_political_corruption", "topic_label", "article_text"] if col in docs.columns]

if SELECT_LABEL is not None:
    display(
        docs[docs[ANALYSIS_LABEL_COLUMN].eq(SELECT_LABEL)]
        .sort_values("analysis_weight", ascending=False)
        [example_cols]
        .head(N_EXAMPLES)
    )
else:
    print("No non-outlier topics available.")

## Export Summary Tables

In [ ]:
EXPORT_DIR = TOPIC_DIR / "inspection_tables"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

topic_totals.to_csv(EXPORT_DIR / "topic_weighted_totals.csv", index=False)
country_topic.to_csv(EXPORT_DIR / "country_topic_shares_top_topics.csv", index=False)
time_topic.to_csv(EXPORT_DIR / "topic_shares_over_time_top_topics.csv", index=False)
country_time.to_csv(EXPORT_DIR / "country_topic_trends_top_topics.csv", index=False)
country_topic_lift.to_csv(EXPORT_DIR / "country_topic_lift_top_topics.csv", index=False)
time_topic_lift.to_csv(EXPORT_DIR / "topic_lift_over_time_top_topics.csv", index=False)
country_time_lift.to_csv(EXPORT_DIR / "country_time_topic_lift_top_topics.csv", index=False)
raw_specificity.to_csv(EXPORT_DIR / "raw_topic_country_specificity.csv", index=False)
raw_to_label.to_csv(EXPORT_DIR / "raw_topic_to_label_mapping.csv", index=False)

print(f"Saved inspection tables under: {EXPORT_DIR}")